# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T0othIess/FlyRank-AI-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*  

**Finding #1 — The Anatomy of Growing Content**

This finding compares pages with rising impressions against pages with falling impressions, and reports that growing pages tend to be longer, younger, and slightly better positioned.
The "growing" vs. "declining" label itself is simple and clearly stated: it's just whether a page's impressions went up or down over the measured window. My question is about the
validation design rather than the label: this is a single-snapshot comparison of two existing groups, not a before/after test. The paper's own report cards ("Expand thin pages... Expected:
improves the odds...") read as action recommendations, but the underlying evidence only shows what growing pages already look like today — it doesn't show that expanding a page *causes* it
to start growing. I'd want to see this framed more explicitly as decision-support ("worth trying, based on what growing pages tend to look like") rather than an expected-outcome claim.    

**Finding #4 — The Freshness Multiplier**

This finding reports a "3.2x health boost" and "57x more impressions" for refreshed old content, based on splitting pages into freshness-age buckets and comparing average outcomes per bucket. The paper is upfront that its `361+` bucket is unstable — the eye-catching 283:1 ratio comes from a bucket with only 1 declining page, meaning a single page moving in or out could flip the number entirely. My question: given that the paper already discloses this instability for one bucket, should headline ratios like "57x" always be reported next to their underlying sample size, rather than as a standalone number in the summary stats? A ratio built on a handful of pages is a different kind of evidence than one built on tens of thousands, even when both get printed as a single multiplier.  

In [1]:
#for finding #1
words_growing = 3180
words_declining = 2311
age_growing = 184
age_declining = 230

word_diff = (words_growing - words_declining)/words_declining   
age_diff = (age_declining-age_growing) /age_declining
print(f"growing words are bigger than declining words by {word_diff:%}")
print(f"growing ages are smaller than declining ages by {age_diff:%} ")

#for finding #4
growing = 283
declining_scenarios = [1, 2, 3]
for declining in declining_scenarios:
    ratio = growing/declining
    print(f"{declining} declining page(s) -> ratio {ratio:.0f}:1")

growing words are bigger than declining words by 37.602769%
growing ages are smaller than declining ages by 20.000000% 
1 declining page(s) -> ratio 283:1
2 declining page(s) -> ratio 142:1
3 declining page(s) -> ratio 94:1


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Model B's split in w05 was already client-grouped (`GroupShuffleSplit`), so there is no literal
"before" grouped state inside that notebook to compare against. To test the split-honesty
question this section asks, I built a naive row-level random split (`train_test_split`, no
grouping) as a counterfactual, using the same features and label as Model B, and compared it
side by side (Model C).

On a single split, Model C (naive) scored 75% precision@20, versus Model B (grouped) at 95% —
the opposite of what I expected: the leakage-and-validation guidance predicts that a naive split
should look artificially *better* than a grouped one, since a model can partly memorize
client identity when the same client appears in both train and test.

Because that result contradicted the expectation, I checked whether either single-split number
was even a stable estimate, by rerunning both splits across 5 different seeds, all constrained
to a test-set size close to 20% so that swings couldn't just be explained by test-set size
changing. The result:

| | mean | min | max |
|---|---|---|---|
| Model B (grouped) precision@20 | 56.0% | 40.0% | 95.0% |
| Model C (naive) precision@20 | 74.0% | 65.0% | 85.0% |

Model B's precision swung by 55 points depending purely on which clients landed in the test set,
while every grouped seed showed 0 clients shared between train and test. Model C looked more
stable (a 20-point range), but its client overlap was 33-35 clients on every single seed —
meaning its calmer range isn't evidence of a better model, it's evidence that the same clients
are consistently present on both sides of the split, letting the model partly recognize them.

**Takeaway (safe language):** a single train/test split precision number — whether grouped or
naive — is not reliable evidence on its own for this dataset, because client group sizes are
uneven enough that which clients happen to fall in the test set materially changes the result.
The grouped split's instability is the honest cost of testing on genuinely unseen clients; the
naive split's apparent stability is a symptom of consistent train/test client overlap, not a
sign of a better-validated model. The w05-reported 95%@20 figure should be read as one point in
a 40-95% range (mean ≈56%) rather than a fixed, trustworthy number.

In [2]:
import os
from huggingface_hub import login
import duckdb
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from IPython.display import display
import numpy as np

HF_TOKEN = os.environ.get("HF_TOKEN")
login(HF_TOKEN)
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

fact_content_daily_performance_table = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')")
dim_content_table = con.sql(f"SELECT * from read_parquet('{rel}/dim_content.parquet')")

df = con.sql("""SELECT f.client_hash_id, f.content_hash_id, SUM(f.gsc_clicks) AS total_clicks, SUM(f.gsc_impressions) AS total_impressions,
                SUM(f.gsc_avg_position * f.gsc_impressions) *1.0 / SUM(f.gsc_impressions) AS weighted_avg_position,
                ANY_VALUE(d.search_volume) AS search_volume, ANY_VALUE(d.competition_level) AS competition_level, ANY_VALUE(d.main_intent) AS main_intent
                FROM fact_content_daily_performance_table AS f JOIN dim_content_table AS d USING(content_hash_id)
                WHERE
                    f.gsc_data_available IS TRUE AND d.is_deleted IS FALSE AND f.gsc_avg_position >0 AND d.search_volume IS NOT NULL AND d.competition_level IS NOT NULL
                GROUP BY f.client_hash_id, f.content_hash_id ORDER BY f.client_hash_id, f.content_hash_id""").df()

df = df.query("total_impressions >= 200").reset_index(drop=True)

df["main_intent"] = df["main_intent"].astype("category")

competition_ranks = pd.api.types.CategoricalDtype(categories=["LOW", "MEDIUM", "HIGH"], ordered=True)
df["competition_level"] = df["competition_level"].astype(competition_ranks)
df["position_tier"] = pd.cut(df["weighted_avg_position"], bins=[0,10,20,float("inf")], labels=["page_1", "striking", "page_3_5"])
expected_ctr_per_tier = df.groupby("position_tier")["total_clicks"].sum() / df.groupby("position_tier")["total_impressions"].sum()
df["expected_ctr"] = df["position_tier"].map(expected_ctr_per_tier).astype(float)
df["ctr"] = df["total_clicks"] / df["total_impressions"]
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

truth_threshold = 0.002
df["is_high_gap"] = (df["ctr_gap"] >= truth_threshold).astype(int)
competition_map = {"LOW": 0, "MEDIUM": 1, "HIGH":2}
df["competition_level_enc"] = df["competition_level"].map(competition_map)




ohe = OneHotEncoder(sparse_output=False)
categories = ["position_tier"]
#fit it on the training data to learn the categories and transforms it (meaning it starts filling up the matrix with 1s and 0s)
position_encoded = ohe.fit_transform(df[categories])

#get the column names, reason why it was on ohe not position_encoded because position_encoded is a matrix without column names, ohe creates the column names
position_columns = ohe.get_feature_names_out(categories)

#the reason for index= df.index is to ensure the dataframe doesnt mix up cuz it will be added to df, so it must have it's index
position_df = pd.DataFrame(data=position_encoded, columns=position_columns, index=df.index)
df[position_columns] = position_df

#basically because search_volume is heavily skewed, we use log for it so that the model doesnt depend too much on it basically
#log1p(x) means log(1+x), reason for the 1+ is cuz x can be 0, to not get a math error
df["log_search_volume"] = np.log1p(df["search_volume"])
feature_list = ["weighted_avg_position", "log_search_volume", "competition_level_enc"] + position_columns.tolist()

X = df[feature_list]
y = df["is_high_gap"]



c:\Users\M-H-M-D\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_hash_id"]

seeds_data = {"seed": [], "train_split": [], "test_split": []}
best_seed = None
best_diff = float("inf")
best_fraction = None
for seed in range(1, 51):
    gss_try = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_idx_try, test_idx_try = next(gss_try.split(X, y, groups=groups))
    fraction_try = len(test_idx_try) / len(X)
    diff = abs(fraction_try - 0.2)
    if diff < 0.01:
        seeds_data["seed"].append(seed)
        seeds_data["train_split"].append(f"{len(train_idx_try) / len(X):%}")
        seeds_data["test_split"].append(f"{fraction_try:%}")
        
    if diff < best_diff:
        best_diff = diff
        best_seed = seed
        best_fraction = fraction_try

gss = GroupShuffleSplit(test_size =0.2, n_splits = 1, random_state = best_seed)
train_idx, test_idx = next(gss.split(X,y, groups= groups))

train_clients = set(df.iloc[train_idx]["client_hash_id"])
test_clients = set(df.iloc[test_idx]["client_hash_id"])

print(f"Clients shared by train and test(grouped model): {len(train_clients & test_clients)}")

X_B_train = X.iloc[train_idx]
X_B_test = X.iloc[test_idx]
y_B_train = y.iloc[train_idx]
y_B_test = y.iloc[test_idx]

model_B = LogisticRegression(max_iter=1000, random_state=1)
model_B.fit(X_B_train, y_B_train)
y_B_pred_proba = model_B.predict_proba(X_B_test)[:,1]

test_df_B = df.loc[X_B_test.index].copy()
test_df_B["is_high_gap"] = y_B_test.values
test_df_B["predicted_high_gap"] = y_B_pred_proba
test_df_model_B = test_df_B.sort_values("predicted_high_gap", ascending=False)

Clients shared by train and test(grouped model): 0


In [4]:
from sklearn.model_selection import train_test_split
X_C_train, X_C_test, y_C_train, y_C_test = train_test_split(X, y, test_size=0.2, random_state=best_seed)

train_clients_C = set(df.loc[X_C_train.index, "client_hash_id"])
test_clients_C = set(df.loc[X_C_test.index,"client_hash_id"])

print(f"Clients shared by train and test (ungrouped model): {len(train_clients_C & test_clients_C)}")

model_C = LogisticRegression(max_iter = 100, random_state = 1)
model_C.fit(X_C_train, y_C_train)
y_C_pred_proba = model_C.predict_proba(X_C_test)[:,1]

test_df_C = df.loc[X_C_test.index].copy()
test_df_C["is_high_gap"] = y_C_test
test_df_C["predicted_high_gap"] = y_C_pred_proba

test_df_model_C = test_df_C.sort_values("predicted_high_gap", ascending=False)

data = {"k":[],"Model B (grouped split) Precision@k": [], "Model C (ungrouped split) Precision@k": []}

for k in [20, 50, 100,200,500,1000]:
    data["k"].append(k)
    top_k_model_B = test_df_model_B.head(k)
    model_B_precision_at_k = top_k_model_B["is_high_gap"].sum() / k
    data["Model B (grouped split) Precision@k"].append(model_B_precision_at_k)

    top_k_model_C = test_df_model_C.head(k)
    model_C_precision_at_k = top_k_model_C["is_high_gap"].sum() / k
    data["Model C (ungrouped split) Precision@k"].append(model_C_precision_at_k)

results = pd.DataFrame(data).style.hide(axis="index").format("{:.1%}", subset=["Model B (grouped split) Precision@k", "Model C (ungrouped split) Precision@k"])\
          .set_properties(**{"text-align": "center"})
display(results)


Clients shared by train and test (ungrouped model): 33


k,Model B (grouped split) Precision@k,Model C (ungrouped split) Precision@k
20,95.0%,75.0%
50,78.0%,74.0%
100,72.0%,65.0%
200,67.5%,63.0%
500,62.6%,56.0%
1000,60.5%,53.5%


I expected the ungrouped to give better results but are less honest because the model could just learn the pattern of a certain client that exists in both train and test splits,  
but.. the opposite happened. the grouped split gave better results than the ungrouped split, that raises a question; is the seed i picked for grouped split was giving those results  
by coincidence?  

To answer that question the following code blocks builds a table to show precision@20 for multiple seeds where the split was almost 80/20.  

In [5]:
seeds_data["model_B_precision_at_20"] = []
seeds_data["model_C_precision_at_20"] = []
for seed in seeds_data["seed"]:
    gss = GroupShuffleSplit(test_size =0.2, n_splits = 1, random_state = seed)
    train_idx, test_idx = next(gss.split(X,y, groups= groups))
    X_B_train_test_per_seed = X.iloc[train_idx]
    X_B_test_per_seed = X.iloc[test_idx]
    y_B_train_per_seed = y.iloc[train_idx]
    y_B_test_per_seed = y.iloc[test_idx]
    model_B_per_seed = LogisticRegression(max_iter=1000, random_state=1)
    model_B_per_seed.fit(X_B_train_test_per_seed, y_B_train_per_seed)
    y_B_pred_proba = model_B_per_seed.predict_proba(X_B_test_per_seed)[:,1]

    test_df_B_per_seed = df.loc[X_B_test_per_seed.index].copy()
    test_df_B_per_seed["is_high_gap"] = y_B_test_per_seed.values
    test_df_B_per_seed["predicted_high_gap"] = y_B_pred_proba
    test_df_model_B_per_seed = test_df_B_per_seed.sort_values("predicted_high_gap", ascending=False)
    top_20_B_per_seed = test_df_model_B_per_seed.head(20)
    seeds_data["model_B_precision_at_20"].append(top_20_B_per_seed["is_high_gap"].sum() / 20)

    X_C_train_per_seed, X_C_test_per_seed, y_C_train_per_seed, y_C_test_per_seed = train_test_split(X, y, test_size=0.2, random_state=seed)

    train_clients_C_per_seed = set(df.loc[X_C_train_per_seed.index, "client_hash_id"])
    test_clients_C_per_seed = set(df.loc[X_C_test_per_seed.index,"client_hash_id"])
    
    print(f"Clients shared by train and test (ungrouped model) for seed {seed}: {len(train_clients_C_per_seed & test_clients_C_per_seed)}")
    
    model_C_per_seed = LogisticRegression(max_iter = 100, random_state = 1)
    model_C_per_seed.fit(X_C_train_per_seed, y_C_train_per_seed)
    y_C_per_seed_pred_proba = model_C_per_seed.predict_proba(X_C_test_per_seed)[:,1]
    
    test_df_C_per_seed = df.loc[X_C_test_per_seed.index].copy()
    test_df_C_per_seed["is_high_gap"] = y_C_test_per_seed
    test_df_C_per_seed["predicted_high_gap"] = y_C_per_seed_pred_proba
    test_df_model_C_per_seed = test_df_C_per_seed.sort_values("predicted_high_gap", ascending=False)
    top_20_C_per_seed = test_df_model_C_per_seed.head(20)
    seeds_data["model_C_precision_at_20"].append(top_20_C_per_seed["is_high_gap"].sum() / 20)

results_table = pd.DataFrame(seeds_data).style.hide(axis="index").format("{:.1%}", subset=["model_B_precision_at_20", "model_C_precision_at_20"]).set_properties(**{"text-align": "center"})
display(results_table)

seed_precision_summary = {}
seed_precision_summary["Model B Seed Precision@20 Summary"] = pd.Series(seeds_data["model_B_precision_at_20"]).describe().loc[["mean", "min", "max"]]
seed_precision_summary["Model C Seed Precision@20 Summary"] = pd.Series(seeds_data["model_C_precision_at_20"]).describe().loc[["mean", "min", "max"]]

display(pd.DataFrame(seed_precision_summary).style.format("{:.2%}", subset=["Model B Seed Precision@20 Summary", "Model C Seed Precision@20 Summary"])\
        .set_properties(**{"text-align": "center"}))

Clients shared by train and test (ungrouped model) for seed 2: 35
Clients shared by train and test (ungrouped model) for seed 27: 34
Clients shared by train and test (ungrouped model) for seed 30: 33
Clients shared by train and test (ungrouped model) for seed 38: 35
Clients shared by train and test (ungrouped model) for seed 49: 33


seed,train_split,test_split,model_B_precision_at_20,model_C_precision_at_20
2,80.903438%,19.096562%,40.0%,65.0%
27,80.347122%,19.652878%,45.0%,70.0%
30,80.777389%,19.222611%,55.0%,85.0%
38,79.609003%,20.390997%,45.0%,75.0%
49,80.047754%,19.952246%,95.0%,75.0%


,Model B Seed Precision@20 Summary,Model C Seed Precision@20 Summary
mean,56.00%,74.00%
min,40.00%,65.00%
max,95.00%,85.00%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.* 

from the table we got in w05 about feature coefficients:  
|                      |coefficient|
|----------------------|-----------|
|position_tier_page_3_5| -6.911404  |
|competition_level_enc | -0.096054 |
|weighted_avg_position | 0.048802  |
|log_search_volume	   | 0.060836  |
|position_tier_striking| 1.956145  |
|position_tier_page_1  | 1.979352  |

**Step 1 — noticing something off**  
Model B's coefficients (from w05) show a large gap between feature groups: the three `position_tier` one-hot columns (`page_1`: +1.98, `striking`: +1.96, `page_3_5`: −6.91) dwarf `weighted_avg_position` (+0.05), `log_search_volume` (+0.06), and `competition_level_enc` (−0.10). One feature category towering over the rest like this, combined with a suspiciously high 95% precision@20, is exactly the symptom the leakage skill calls out — worth investigating before trusting it.  

**Step 2 — testing the suspect**  
Retraining on the same split with the `position_tier` one-hot columns removed (Model D) collapsed precision@k from 95.0%/78.0%/72.0% (k=20/50/100) down to roughly 40-43% across every k — landing right at the 42.6% base rate. `position_tier` is responsible for essentially all of Model B's apparent skill.  

**Step 3 — the root cause**  
`is_high_gap` is derived from `ctr_gap = expected_ctr - ctr`, where `expected_ctr` is a benchmark looked up per `position_tier`. `expected_ctr_per_tier` shows `page_3_5 = 0.001364`, below the label's `truth_threshold` of `0.002`. Since actual `ctr` can never go below 0, the maximum possible `ctr_gap` for any `page_3_5` page is `0.001364` — no `page_3_5` page can ever be labeled `is_high_gap = 1`, regardless of its real performance. The observed `is_high_gap` rate within `page_3_5` is exactly `0.000000`, confirming this. So `position_tier`'s dominance isn't a sign of a uniquely strong feature — it reveals the label by construction: `position_tier` helped build part of the label formula, and the model is partly reading that back rather than learning an independent pattern.  

**Failure examples**  
The `page_3_5` cap isn't just a statistical artifact — it silences pages with real signal. For example, `content_ff04f85aa152b2a0` recorded 2,328 impressions and 0 clicks — a page clearly underperforming by any plain reading — yet it's capped at `is_high_gap = 0` purely because of its tier, alongside every other `page_3_5` page regardless of how it actually performed.  

**Second, smaller risk — benchmark computed across train and test**  
`expected_ctr_per_tier` was originally computed by grouping the entire dataset (train and test rows together) before the split. I tested whether recomputing it using only training rows would change the label: 3,117 of ~82,507 rows (≈3.8%) flipped from `is_high_gap = 0` to `1` — entirely within `page_1` and `striking` (no row went the other direction, and `page_3_5` was unaffected either way, since it stays capped regardless of which population built the benchmark). Refitting the model on this train-only label (Model E) barely changed precision@20/50 and modestly *increased* precision at higher k (k=1000: 65.4% vs. Model B's 60.5%) — confirming this is a real but minor contributor next to the `position_tier` issue, not the main driver of Model B's inflated score.  

**What I'd change, without rebuilding the model here:**  
the threshold for `is_high_gap` should be relative to each tier's own `expected_ctr` rather than one flat absolute cutoff applied to all tiers, and `expected_ctr_per_tier` should be computed using only training rows. Both changes are out of scope for this audit notebook, but should be treated as required before Model B's precision numbers are used for anything beyond this internal report.  

In [35]:
feature_list_D = ["weighted_avg_position", "log_search_volume", "competition_level_enc"]
X_D = df[feature_list_D]
y_D = y

train_D_idx, test_D_idx = next(gss.split(X_D,y_D, groups= groups))

X_D_train = X_D.iloc[train_D_idx]
X_D_test = X_D.iloc[test_D_idx]
y_D_train = y_D.iloc[train_D_idx]
y_D_test = y_D.iloc[test_D_idx]

model_D = LogisticRegression(max_iter=1000, random_state=1)
model_D.fit(X_D_train, y_D_train)
y_D_pred_proba = model_D.predict_proba(X_D_test)[:,1]

test_df_D = df.loc[X_D_test.index].copy()
test_df_D["is_high_gap"] = y_D_test.values
test_df_D["predicted_high_gap"] = y_D_pred_proba
test_df_model_D = test_df_D.sort_values("predicted_high_gap", ascending=False)

test_data = {"k": data["k"], "Model B Precision@k": data["Model B (grouped split) Precision@k"], "Model D (no position_tier) Precision@k": []}
for k in [20, 50, 100,200,500,1000]:
    top_k_model_D = test_df_model_D.head(k)
    model_D_precision_at_k = top_k_model_D["is_high_gap"].sum() / k
    test_data["Model D (no position_tier) Precision@k"].append(model_D_precision_at_k)

test_results = pd.DataFrame(test_data).style.hide(axis="index").format("{:.1%}", subset=["Model B Precision@k", "Model D (no position_tier) Precision@k"])\
          .set_properties(**{"text-align": "center"})
display(test_results)

display(expected_ctr_per_tier.to_frame(name="Expected ctr per tier").reset_index())

display(test_df_B.groupby("position_tier")["is_high_gap"].mean().to_frame(name="is_high_gap mean per position").reset_index())

print(f'Is expected ctr for position "page_3_5" below the truth threshold({truth_threshold})? {expected_ctr_per_tier["page_3_5"] < truth_threshold}')

#expected_ctr leakage check
expected_ctr_per_tier_v2 = df.loc[X_B_train.index].groupby("position_tier")["total_clicks"].sum() / df.loc[X_B_train.index].groupby("position_tier")["total_impressions"].sum()
df["expected_ctr_v2"] = df["position_tier"].map(expected_ctr_per_tier_v2).astype(float)
df["ctr_gap_v2"] = df["expected_ctr_v2"] - df["ctr"]
df["is_high_gap_v2"] = (df["ctr_gap_v2"] >= truth_threshold).astype(int)

X_E_train = X_B_train
X_E_test = X_B_test
y_E_train = df.loc[X_E_train.index, "is_high_gap_v2"]
y_E_test = df.loc[X_E_test.index, "is_high_gap_v2"]

model_E = LogisticRegression(max_iter=1000, random_state=1)
model_E.fit(X_E_train, y_E_train)
y_E_pred_proba = model_E.predict_proba(X_E_test)[:,1]

test_df_E = df.loc[X_E_test.index].copy()
test_df_E["is_high_gap_v2"] = y_E_test.values
test_df_E["predicted_high_gap"] = y_E_pred_proba
test_df_model_E = test_df_E.sort_values("predicted_high_gap", ascending=False)

test_data["Model E Precision@k"] = []
for k in [20, 50, 100,200,500,1000]:
    top_k_model_E = test_df_model_E.head(k)
    model_E_precision_at_k = top_k_model_E["is_high_gap_v2"].sum() / k
    test_data["Model E Precision@k"].append(model_E_precision_at_k)


test_results = pd.DataFrame(test_data)[["k","Model B Precision@k", "Model E Precision@k"]].style.hide(axis="index").format("{:.1%}", subset=["Model B Precision@k", "Model E Precision@k"])\
          .set_properties(**{"text-align": "center"})
display(test_results)

display(df.query("is_high_gap != is_high_gap_v2")[["is_high_gap", "is_high_gap_v2"]].value_counts())
display(test_df_E.groupby("position_tier")["is_high_gap_v2"].mean().to_frame(name="is_high_gap_v2 mean per position").reset_index())

columns = ["content_hash_id","total_clicks", "total_impressions", "weighted_avg_position", "position_tier", "search_volume", "competition_level", "ctr", "ctr_gap", "is_high_gap_v2"]
df.query("position_tier == 'page_3_5'")[columns].sort_values("ctr_gap", ascending=False).head(5).style.set_properties(**{"text-align": "center"})


k,Model B Precision@k,Model D (no position_tier) Precision@k
20,95.0%,40.0%
50,78.0%,40.0%
100,72.0%,40.0%
200,67.5%,43.0%
500,62.6%,42.6%
1000,60.5%,38.0%


,position_tier,Expected ctr per tier
0,page_1,0.003376
1,striking,0.003182
2,page_3_5,0.001364


,position_tier,is_high_gap mean per position
0,page_1,0.448339
1,striking,0.587500
2,page_3_5,0.000000


Is expected ctr for position "page_3_5" below the truth threshold(0.002)? True


k,Model B Precision@k,Model E Precision@k
20,95.0%,95.0%
50,78.0%,78.0%
100,72.0%,71.0%
200,67.5%,73.0%
500,62.6%,68.2%
1000,60.5%,65.4%


is_high_gap  is_high_gap_v2
0            1                 3117
Name: count, dtype: int64

,position_tier,is_high_gap_v2 mean per position
0,page_1,0.516528
1,striking,0.602000
2,page_3_5,0.000000


,content_hash_id,total_clicks,total_impressions,weighted_avg_position,position_tier,search_volume,competition_level,ctr,ctr_gap,is_high_gap_v2
82506,content_fff61fa922a978fb,0.000000,480.000000,41.197917,page_3_5,10,LOW,0.000000,0.001364,0
6,content_a0a6b37ae2f9a09c,0.000000,248.000000,21.725806,page_3_5,0,LOW,0.000000,0.001364,0
82502,content_ff04f85aa152b2a0,0.000000,2328.000000,70.912371,page_3_5,10,LOW,0.000000,0.001364,0
82501,content_fea4ddbaf749a6cb,0.000000,1752.000000,22.642123,page_3_5,0,LOW,0.000000,0.001364,0
16,content_0050d70972c911bf,0.000000,370.000000,52.583784,page_3_5,210,LOW,0.000000,0.001364,0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (W05):** "Model B reaches 95.0% precision@20 and 72.0% precision@100, meaning 19 of the top 20 pages and 72 of the top 100 pages are true high-gap pages."

**Why this overreaches:** two separate issues undermine this number. First, it comes from a single grouped split — across five grouped seeds, precision@20 ranged from 40% to 95% (mean 56%), so the single-split 95% is not a stable estimate. Second, the model relies almost entirely on `position_tier`, which was also used to help construct the label itself (`expected_ctr` is looked up per tier) — with `position_tier` removed, precision collapses to roughly the base rate. A related effect of this same construction: `page_3_5` pages can never be labeled high-gap under any circumstance, regardless of real performance.

**Safe rewrite:** On the grouped split used in W05, Model B measured 95% precision@20, but this figure ranged from 40% to 95% across five grouped seeds (mean 56%), and most of that measured skill traces back to `position_tier`, a feature tied to the label's own construction — not an independently learned pattern.  
These results should not be treated as a stable decision-support tool without further validation — across five grouped seeds, measured precision@20 ranged from below the 42.6% base rate (40%) up to 95%, meaning the model's real-world reliability for prioritizing `page_1`/`striking` pages is currently unresolved, not established. It should not be used to make any claim about `page_3_5` content at all, since the label cannot register that tier as high-gap under the current design.  

In [39]:

display(pd.DataFrame(seeds_data)[["seed","train_split","test_split","model_B_precision_at_20"]].style.hide(axis="index")
        .format("{:.1%}", subset=["model_B_precision_at_20"]).set_properties(**{"text-align": "center"})
       )

display(pd.DataFrame(seed_precision_summary)["Model B Seed Precision@20 Summary"].to_frame("Model B Seed Precision@20 Summary")
        .style.format("{:.2%}", subset=["Model B Seed Precision@20 Summary"]).set_properties(**{"text-align": "center"})
        )

seed,train_split,test_split,model_B_precision_at_20
2,80.903438%,19.096562%,40.0%
27,80.347122%,19.652878%,45.0%
30,80.777389%,19.222611%,55.0%
38,79.609003%,20.390997%,45.0%
49,80.047754%,19.952246%,95.0%


,Model B Seed Precision@20 Summary
mean,56.00%
min,40.00%
max,95.00%


## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.